# v8.5 — double-buffered KV pipeline — the gate (CUDA-core, Colab T4)

v8 measured the decode kernel at only **~10-11% HBM** (per-CTA-bound, not bandwidth-bound). v8 Cut 1's hot loop STALLS on every KV tile (`load → sync → compute → sync → load`), so the global-load latency is exposed. **v8.5 (`v8_gqa_db`)** software-pipelines it: prefetch tile *N+1* while computing tile *N*, so the load latency hides behind the warp-shuffle compute. Portable double-buffer (ordinary `ld.global`, NOT `cp.async` which is Ampere-only); two HALF KV buffers = Cut 1's single FP32 buffer, so occupancy (2 blocks/SM) is unchanged and the ONLY new variable is the load/compute overlap.

**Headline check: does double-buffering raise `%HBM` (toward bandwidth-bound) and lower µs/tok vs Cut 1 (`v8_gqa`)?** If `%HBM` doesn't move, the stall is the GEMV reduction, not the load — also a finding.

## 0. Dependencies + GPU (venv-safe)

In [ ]:
import os, sys, subprocess

def pip(*pkgs, extra=()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra], check=True)

# 1) Physical GPU on this runtime? (Colab defaults to CPU; pick a GPU explicitly.)
try:
    has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
except FileNotFoundError:
    has_gpu = False
if not has_gpu:
    raise SystemExit(
        'No GPU on this Colab runtime. FIX: Runtime > Change runtime type > T4 GPU > Save, '
        'then Runtime > Restart session, then re-run from the top. (This kernel needs a Turing T4.)')

# 2) Install deps (incl. numpy) BEFORE importing torch, so torch's numpy bridge initializes --
#    importing torch first on a numpy-less venv (vast.ai) prints 'Failed to initialize NumPy'.
pip('ninja', 'pytest', 'numpy')

# 3) torch present AND CUDA-enabled? A CPU-only wheel raises "not compiled with CUDA" on any kernel.
try:
    import torch
    cuda_ok = torch.cuda.is_available()
except ImportError:
    torch, cuda_ok = None, False

if not cuda_ok:
    pip('torch', extra=('--index-url', 'https://download.pytorch.org/whl/cu124'))
    raise SystemExit(
        'A GPU is present but torch was a CPU-only build -- installed the CUDA build. NOW: '
        'restart the kernel/session, then re-run this cell (the old CPU torch stays loaded until restart).')

# vast.ai/venv: !-cells spawn a bare shell without the venv on PATH -> `python` not found.
os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')

print('torch', torch.__version__, '| cuda', torch.version.cuda, '| cap', torch.cuda.get_device_capability())
!nvidia-smi --query-gpu=name,compute_cap --format=csv
!which python && python -c "import torch; print('shell python sees torch', torch.__version__)"

## 1. Get the repo

In [ ]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works
import os, sys, subprocess
if os.path.basename(os.getcwd()) != 'flashattention-cuda':
    if not os.path.isdir('flashattention-cuda'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('flashattention-cuda')
subprocess.run(['git', 'pull', 'origin', 'main'])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

## 2. Roofline — UNCHANGED (`AI=2G/b`, same floor); the prediction is a SCHEDULE claim

Double-buffering moves the same bytes → the model is identical to v8. The prediction the model CAN'T express: %HBM should climb from ~11% toward the floor as the prefetch hides the load latency. Success = %HBM up + µs/tok down vs Cut 1; null (no movement) = the bottleneck is the per-key shuffle reduction, not the load.

In [ ]:
from roofline.archs import get_arch
from roofline.model import estimate
arch = get_arch('sm_75')   # the T4 we're running on
print('arch:', arch.name, '| HBM', arch.hbm_bw_gbps, 'GB/s')
print(f"{'G':>3} | {'AI=2G/b':>8} | {'limiter':>7} | {'t_hbm floor':>12}")
for G in (1,2,4,8,16,32):
    e = estimate(arch, B=8, H=8, N_q=1, N_k=8192, d=128, precision='fp16', G=G)
    print(f'{G:>3} | {e.arithmetic_intensity:8.1f} | {e.limiter.upper():>7} | {e.t_hbm*1e3:9.4f}ms')
print('\nPrediction: same AI/floor as v8; double-buffering should RAISE %HBM + LOWER us/tok vs Cut 1.')


## 3. Build v8_gqa_db (JIT)

In [ ]:
import glob, os, shutil
for d in glob.glob(os.path.expanduser('~/.cache/torch_extensions/*/fa_v8_gqa_db')):
    if not glob.glob(os.path.join(d, '*.so')):
        shutil.rmtree(d, ignore_errors=True); print('cleaned stale build:', d)
from bindings.load import build_kernel
mod = build_kernel('v8_gqa_db')
print('built:', mod)


## 4. Correctness gate — v8_gqa_db (Gate 1 of 2)

Same GQA cases as Cut 1 (decode `G∈{1,2,4,8}` × non-multiple `N_k` × causal+offset, idle/pad `G=3` + multi-tile `G=16`, square reduction). Oracle = `sdpa_reference_gqa`, tol 2e-2.

In [ ]:
!python -m pytest tests/test_correctness.py -k "v8_gqa_db" -q


## 5. THE A/B — v8_gqa_db (double-buffered) vs v8_gqa (Cut 1), same G-sweep

Watch the **%HBM** column (did the prefetch close the per-CTA gap?) and **µs/tok**.

In [ ]:
print('=== v8.5: v8_gqa_db (double-buffered) ===')
!python -m bench.harness --backend v8_gqa_db --decode --seq 8192 --heads 32 --gqa-group 1 2 4 8 16 32
print('\n=== Cut 1: v8_gqa (single-buffer) — same workload ===')
!python -m bench.harness --backend v8_gqa --decode --seq 8192 --heads 32 --gqa-group 1 2 4 8 16 32


## 6. Reclaim-SDPA-at-batch (G=8) — double-buffer must still beat SDPA

In [ ]:
print('=== v8_gqa_db (double-buffered) ===')
!python -m bench.harness --backend v8_gqa_db --decode --seq 8192 --heads 8 --gqa-group 8 --batch-sweep 1 8 16 32 64
print('\n=== v8_gqa (Cut 1) ===')
!python -m bench.harness --backend v8_gqa --decode --seq 8192 --heads 8 --gqa-group 8 --batch-sweep 1 8 16 32 64
